# NLP Model Training and Evaluation

This notebook trains and evaluates multiple NLP/ML models for customer satisfaction prediction.

**Target**: `csat_score` binarized (Positive: 4-5, Negative: 1-3)

**Models**:
1. TF-IDF + Logistic Regression (Baseline)
2. TF-IDF + Random Forest
3. XGBoost (Structured Features)
4. Combined Model (NLP + Structured Features)

## Section 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

print('All libraries loaded successfully.')

In [ ]:
# Load the raw dataset
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')

# Parse datetime columns
df['issue_reported_at'] = pd.to_datetime(df['issue_reported_at'], errors='coerce', dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)

# Create binary target: Positive (4-5) vs Negative (1-3)
df['target'] = (df['csat_score'] >= 4).astype(int)
print(f'\nTarget distribution:')
print(f'  Positive (CSAT 4-5): {df["target"].sum()} ({df["target"].mean()*100:.1f}%)')
print(f'  Negative (CSAT 1-3): {(1-df["target"]).sum()} ({(1-df["target"].mean())*100:.1f}%)')

## Section 2: Feature Engineering (Corrected)

Only using features with sufficient coverage (>10% non-null). Columns with 99%+ missing are excluded.

In [ ]:
# --- NLP Text Preprocessing ---
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = text.lower()
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

print('Preprocessing text...')
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Done. Non-empty messages: {(df["cleaned_message"] != "").sum()}')

In [ ]:
# --- Structured Features (100% coverage) ---

# Response time in minutes
df['response_time_minutes'] = (
    (df['issue_responded'] - df['issue_reported_at']).dt.total_seconds() / 60
).clip(lower=0).fillna(0)

# Hour and day of week
df['issue_hour'] = df['issue_reported_at'].dt.hour.fillna(0).astype(int)
df['issue_day_of_week'] = df['issue_reported_at'].dt.dayofweek.fillna(0).astype(int)

# Encode categorical features
le_channel = LabelEncoder()
df['channel_encoded'] = le_channel.fit_transform(df['channel_name'].fillna('Unknown'))

le_category = LabelEncoder()
df['category_encoded'] = le_category.fit_transform(df['category'].fillna('Unknown'))

le_subcategory = LabelEncoder()
df['subcategory_encoded'] = le_subcategory.fit_transform(df['sub-category'].fillna('Unknown'))

le_shift = LabelEncoder()
df['shift_encoded'] = le_shift.fit_transform(df['agent_shift'].fillna('Unknown'))

tenure_map = {'On Job Training': 0, '0-30': 1, '31-60': 2, '61-90': 3, '>90': 4}
df['tenure_encoded'] = df['tenure_bucket'].map(tenure_map).fillna(0).astype(int)

# Text-derived features
df['has_message'] = (df['cleaned_message'] != '').astype(int)
df['cleaned_word_count'] = df['cleaned_message'].apply(lambda x: len(x.split()) if x else 0)

# Define structured feature columns
structured_features = [
    'response_time_minutes', 'issue_hour', 'issue_day_of_week',
    'channel_encoded', 'category_encoded', 'subcategory_encoded',
    'shift_encoded', 'tenure_encoded', 'message_length', 'word_count',
    'has_message', 'cleaned_word_count'
]

print(f'Structured features: {len(structured_features)}')
print(structured_features)

## Section 3: Train-Test Split

In [ ]:
# Split data
X_structured = df[structured_features].fillna(0)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_structured, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTarget distribution (train): Positive={y_train.mean()*100:.1f}%')
print(f'Target distribution (test): Positive={y_test.mean()*100:.1f}%')

# TF-IDF for text features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)

# Use indices to align text with structured features
train_idx = X_train.index
test_idx = X_test.index

X_text_train = tfidf.fit_transform(df.loc[train_idx, 'cleaned_message'])
X_text_test = tfidf.transform(df.loc[test_idx, 'cleaned_message'])

print(f'\nTF-IDF features: {X_text_train.shape[1]}')
print(f'Vocabulary sample: {list(tfidf.vocabulary_.keys())[:10]}')

## Section 4: Model 1 — TF-IDF + Logistic Regression (Baseline)

In [ ]:
# Model 1: TF-IDF + Logistic Regression
print('=== Model 1: TF-IDF + Logistic Regression ===')
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_text_train, y_train)

y_pred_lr = lr_model.predict(X_text_test)
y_prob_lr = lr_model.predict_proba(X_text_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_lr)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_lr)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_lr)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

## Section 5: Model 2 — TF-IDF + Random Forest

In [ ]:
# Model 2: TF-IDF + Random Forest
print('=== Model 2: TF-IDF + Random Forest ===')
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=20, random_state=42,
    class_weight='balanced', n_jobs=-1
)
rf_model.fit(X_text_train, y_train)

y_pred_rf = rf_model.predict(X_text_test)
y_prob_rf = rf_model.predict_proba(X_text_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_rf)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_rf)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Negative', 'Positive']))

## Section 6: Model 3 — XGBoost (Structured Features)

In [ ]:
# Model 3: XGBoost on structured features
print('=== Model 3: XGBoost (Structured Features) ===')

# Calculate scale_pos_weight for imbalanced classes
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_weight, random_state=42,
    eval_metric='logloss', use_label_encoder=False
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_xgb):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_xgb, target_names=['Negative', 'Positive']))

In [ ]:
# Feature importance from XGBoost
importance = pd.DataFrame({
    'feature': structured_features,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('XGBoost Feature Importance', fontsize=14)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
print(importance.to_string(index=False))

## Section 7: Model 4 — Combined (NLP + Structured Features)

In [ ]:
# Model 4: Combined NLP + Structured features
from scipy.sparse import hstack

print('=== Model 4: Combined (TF-IDF + Structured) + Logistic Regression ===')

# Combine TF-IDF and structured features
from scipy.sparse import csr_matrix
X_struct_train_sparse = csr_matrix(X_train.values)
X_struct_test_sparse = csr_matrix(X_test.values)

X_combined_train = hstack([X_text_train, X_struct_train_sparse])
X_combined_test = hstack([X_text_test, X_struct_test_sparse])

print(f'Combined feature matrix: {X_combined_train.shape[1]} features')

# Train combined Logistic Regression
combined_lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
combined_lr.fit(X_combined_train, y_train)

y_pred_combined = combined_lr.predict(X_combined_test)
y_prob_combined = combined_lr.predict_proba(X_combined_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_combined)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_combined)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_combined)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_combined)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_combined):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_combined, target_names=['Negative', 'Positive']))

## Section 8: Model Comparison Summary

In [ ]:
# Model comparison
results = pd.DataFrame({
    'Model': [
        'TF-IDF + Logistic Regression',
        'TF-IDF + Random Forest',
        'XGBoost (Structured)',
        'Combined (NLP + Structured)'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_combined)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_combined)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_combined)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_combined)
    ],
    'AUC-ROC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb),
        roc_auc_score(y_test, y_prob_combined)
    ]
})

# Format as percentages
print('=== MODEL COMPARISON ===')
display_results = results.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    display_results[col] = (display_results[col] * 100).round(2).astype(str) + '%'
display_results['AUC-ROC'] = display_results['AUC-ROC'].round(4)
print(display_results.to_string(index=False))

# Best model
best_idx = results['F1-Score'].idxmax()
print(f'\n🏆 Best Model (by F1-Score): {results.loc[best_idx, "Model"]}')
print(f'   Accuracy: {results.loc[best_idx, "Accuracy"]*100:.2f}%')
print(f'   F1-Score: {results.loc[best_idx, "F1-Score"]*100:.2f}%')
print(f'   AUC-ROC: {results.loc[best_idx, "AUC-ROC"]:.4f}')

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of metrics
metrics_plot = results.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
metrics_plot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='black', width=0.7)
axes[0].set_title('Model Performance Comparison', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=15)

# Confusion matrix for best model
best_preds = [y_pred_lr, y_pred_rf, y_pred_xgb, y_pred_combined][best_idx]
cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
axes[1].set_title(f'Confusion Matrix — {results.loc[best_idx, "Model"]}', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Save the best model
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(xgb_model, '../models/xgboost_csat_model.pkl')
joblib.dump(combined_lr, '../models/combined_lr_model.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

print('Models saved to ../models/')
print('  - xgboost_csat_model.pkl')
print('  - combined_lr_model.pkl')
print('  - tfidf_vectorizer.pkl')

In [ ]:
# Load the raw dataset
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')

# Parse datetime columns
df['issue_reported_at'] = pd.to_datetime(df['issue_reported_at'], errors='coerce', dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)

# Create binary target: Positive (4-5) vs Negative (1-3)
df['target'] = (df['csat_score'] >= 4).astype(int)
print(f'\nTarget distribution:')
print(f'  Positive (CSAT 4-5): {df["target"].sum()} ({df["target"].mean()*100:.1f}%)')
print(f'  Negative (CSAT 1-3): {(1-df["target"]).sum()} ({(1-df["target"].mean())*100:.1f}%)')

## Section 2: Feature Engineering (Corrected)

Only using features with sufficient coverage (>10% non-null). Columns with 99%+ missing are excluded.

In [ ]:
# --- NLP Text Preprocessing ---
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = text.lower()
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

print('Preprocessing text...')
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Done. Non-empty messages: {(df["cleaned_message"] != "").sum()}')

In [ ]:
# --- Structured Features (100% coverage) ---

# Response time in minutes
df['response_time_minutes'] = (
    (df['issue_responded'] - df['issue_reported_at']).dt.total_seconds() / 60
).clip(lower=0).fillna(0)

# Hour and day of week
df['issue_hour'] = df['issue_reported_at'].dt.hour.fillna(0).astype(int)
df['issue_day_of_week'] = df['issue_reported_at'].dt.dayofweek.fillna(0).astype(int)

# Encode categorical features
le_channel = LabelEncoder()
df['channel_encoded'] = le_channel.fit_transform(df['channel_name'].fillna('Unknown'))

le_category = LabelEncoder()
df['category_encoded'] = le_category.fit_transform(df['category'].fillna('Unknown'))

le_subcategory = LabelEncoder()
df['subcategory_encoded'] = le_subcategory.fit_transform(df['sub-category'].fillna('Unknown'))

le_shift = LabelEncoder()
df['shift_encoded'] = le_shift.fit_transform(df['agent_shift'].fillna('Unknown'))

tenure_map = {'On Job Training': 0, '0-30': 1, '31-60': 2, '61-90': 3, '>90': 4}
df['tenure_encoded'] = df['tenure_bucket'].map(tenure_map).fillna(0).astype(int)

# Text-derived features
df['has_message'] = (df['cleaned_message'] != '').astype(int)
df['cleaned_word_count'] = df['cleaned_message'].apply(lambda x: len(x.split()) if x else 0)

# Define structured feature columns
structured_features = [
    'response_time_minutes', 'issue_hour', 'issue_day_of_week',
    'channel_encoded', 'category_encoded', 'subcategory_encoded',
    'shift_encoded', 'tenure_encoded', 'message_length', 'word_count',
    'has_message', 'cleaned_word_count'
]

print(f'Structured features: {len(structured_features)}')
print(structured_features)

## Section 3: Train-Test Split

In [ ]:
# Split data
X_structured = df[structured_features].fillna(0)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_structured, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTarget distribution (train): Positive={y_train.mean()*100:.1f}%')
print(f'Target distribution (test): Positive={y_test.mean()*100:.1f}%')

# TF-IDF for text features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)

# Use indices to align text with structured features
train_idx = X_train.index
test_idx = X_test.index

X_text_train = tfidf.fit_transform(df.loc[train_idx, 'cleaned_message'])
X_text_test = tfidf.transform(df.loc[test_idx, 'cleaned_message'])

print(f'\nTF-IDF features: {X_text_train.shape[1]}')
print(f'Vocabulary sample: {list(tfidf.vocabulary_.keys())[:10]}')

## Section 4: Model 1 — TF-IDF + Logistic Regression (Baseline)

In [ ]:
# Model 1: TF-IDF + Logistic Regression
print('=== Model 1: TF-IDF + Logistic Regression ===')
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_text_train, y_train)

y_pred_lr = lr_model.predict(X_text_test)
y_prob_lr = lr_model.predict_proba(X_text_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_lr)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_lr)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_lr)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

## Section 5: Model 2 — TF-IDF + Random Forest

In [ ]:
# Model 2: TF-IDF + Random Forest
print('=== Model 2: TF-IDF + Random Forest ===')
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=20, random_state=42,
    class_weight='balanced', n_jobs=-1
)
rf_model.fit(X_text_train, y_train)

y_pred_rf = rf_model.predict(X_text_test)
y_prob_rf = rf_model.predict_proba(X_text_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_rf)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_rf)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Negative', 'Positive']))

## Section 6: Model 3 — XGBoost (Structured Features)

In [ ]:
# Model 3: XGBoost on structured features
print('=== Model 3: XGBoost (Structured Features) ===')

# Calculate scale_pos_weight for imbalanced classes
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_weight, random_state=42,
    eval_metric='logloss', use_label_encoder=False
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_xgb)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_xgb):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_xgb, target_names=['Negative', 'Positive']))

In [ ]:
# Feature importance from XGBoost
importance = pd.DataFrame({
    'feature': structured_features,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('XGBoost Feature Importance', fontsize=14)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
print(importance.to_string(index=False))

## Section 7: Model 4 — Combined (NLP + Structured Features)

In [ ]:
# Model 4: Combined NLP + Structured features
from scipy.sparse import hstack

print('=== Model 4: Combined (TF-IDF + Structured) + Logistic Regression ===')

# Combine TF-IDF and structured features
from scipy.sparse import csr_matrix
X_struct_train_sparse = csr_matrix(X_train.values)
X_struct_test_sparse = csr_matrix(X_test.values)

X_combined_train = hstack([X_text_train, X_struct_train_sparse])
X_combined_test = hstack([X_text_test, X_struct_test_sparse])

print(f'Combined feature matrix: {X_combined_train.shape[1]} features')

# Train combined Logistic Regression
combined_lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
combined_lr.fit(X_combined_train, y_train)

y_pred_combined = combined_lr.predict(X_combined_test)
y_prob_combined = combined_lr.predict_proba(X_combined_test)[:, 1]

print(f'\nAccuracy: {accuracy_score(y_test, y_pred_combined)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_combined)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred_combined)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred_combined)*100:.2f}%')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_combined):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_combined, target_names=['Negative', 'Positive']))

## Section 8: Model Comparison Summary

In [ ]:
# Model comparison
results = pd.DataFrame({
    'Model': [
        'TF-IDF + Logistic Regression',
        'TF-IDF + Random Forest',
        'XGBoost (Structured)',
        'Combined (NLP + Structured)'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_combined)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_combined)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_combined)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_combined)
    ],
    'AUC-ROC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb),
        roc_auc_score(y_test, y_prob_combined)
    ]
})

# Format as percentages
print('=== MODEL COMPARISON ===')
display_results = results.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    display_results[col] = (display_results[col] * 100).round(2).astype(str) + '%'
display_results['AUC-ROC'] = display_results['AUC-ROC'].round(4)
print(display_results.to_string(index=False))

# Best model
best_idx = results['F1-Score'].idxmax()
print(f'\n🏆 Best Model (by F1-Score): {results.loc[best_idx, "Model"]}')
print(f'   Accuracy: {results.loc[best_idx, "Accuracy"]*100:.2f}%')
print(f'   F1-Score: {results.loc[best_idx, "F1-Score"]*100:.2f}%')
print(f'   AUC-ROC: {results.loc[best_idx, "AUC-ROC"]:.4f}')

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of metrics
metrics_plot = results.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
metrics_plot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='black', width=0.7)
axes[0].set_title('Model Performance Comparison', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=15)

# Confusion matrix for best model
best_preds = [y_pred_lr, y_pred_rf, y_pred_xgb, y_pred_combined][best_idx]
cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
axes[1].set_title(f'Confusion Matrix — {results.loc[best_idx, "Model"]}', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Save the best model
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(xgb_model, '../models/xgboost_csat_model.pkl')
joblib.dump(combined_lr, '../models/combined_lr_model.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

print('Models saved to ../models/')
print('  - xgboost_csat_model.pkl')
print('  - combined_lr_model.pkl')
print('  - tfidf_vectorizer.pkl')